# Olist Product Recommendation and Customer Retention

This notebook develops CSE303 catalog topic 9.2 using the Olist Brazilian E-Commerce dataset. We will explore customer behavior, recommend relevant products, and estimate repeat-purchase likelihood. The verified pipeline will later support an application with a conversational assistant.

The proposed application user is a marketing employee exploring customer retention opportunities. The prediction horizon and application technology will be decided after assessing the data.

We will develop one section at a time and explain its actual results in [PIPELINE_NOTES.md](PIPELINE_NOTES.md). See [PROJECT_PLAN.md](PROJECT_PLAN.md) for the broader sequence.


## References

- [Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) provides the source tables.
- [Mohab Allam's reference notebook](https://www.kaggle.com/code/mohabovic/eda-satisfaction-prediction-nlp-rfm-deployment/notebook) informs our approach to data preparation, EDA, satisfaction analysis, and segmentation. Specific adaptations will be credited as they are introduced.
- The course kickoff guide and Week 10 recommender starter inform the deliverables and understanding checks.

This first section contains our own loading code.


## 1 Load the source tables

Read all nine CSV files and record their dimensions. Keep the tables separate so that their row meanings and relationships can be inspected before joining them.

Open this notebook in VS Code and select a Python kernel with pandas installed. Run the cells from top to bottom with the repository root as the working directory. Source CSVs should be directly inside `data/raw/`.

`Path` constructs filesystem paths. pandas loads CSV files into DataFrames, which are tables with named columns.


In [1]:
from pathlib import Path

import pandas as pd

print(f"pandas version: {pd.__version__}")


pandas version: 3.0.1


### Locate and check the files

`data_files` maps meaningful table names to the original CSV filenames. Check that the source folder and every required file exist before loading. Relative paths let collaborators run the same code from their own repository root.


In [2]:
project_root = Path.cwd()
data_dir = project_root / "data" / "raw"

if not data_dir.is_dir():
    raise FileNotFoundError(
        "Expected data/raw/ under the working directory. "
        "Run the notebook from the repository root and check the README download instructions."
    )

data_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    filename
    for filename in data_files.values()
    if not (data_dir / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(f"Missing source files: {', '.join(missing_files)}")

print(f"Found all {len(data_files)} required CSV files in data/raw/.")


Found all 9 required CSV files in data/raw/.


### Read the tables and summarize their dimensions

`pd.read_csv()` reads each file into a DataFrame. The `tables` dictionary holds them under meaningful names, such as `tables["customers"]` and `tables["orders"]`.

Postal-code prefixes are identifiers. Loading them with the `string` dtype preserves leading zeros when present in the source. Other types are initially inferred by pandas. Timestamps will be inspected and parsed in a later section.

`.shape[0]` returns row count, and `.shape[1]` returns column count. These counts describe the source tables before cleaning.


In [3]:
postal_code_types = {
    "customer_zip_code_prefix": "string",
    "seller_zip_code_prefix": "string",
    "geolocation_zip_code_prefix": "string",
}

tables = {}

for table_name, filename in data_files.items():
    tables[table_name] = pd.read_csv(
        data_dir / filename,
        dtype=postal_code_types,
    )

summary_rows = []

for table_name, dataframe in tables.items():
    summary_rows.append({
        "table": table_name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
    })

data_summary = pd.DataFrame(summary_rows)

print(data_summary.to_string(index=False))
print()
print(f"Loaded {len(tables)} source tables.")


               table    rows  columns
           customers   99441        5
              orders   99441        8
         order_items  112650        7
      order_payments  103886        5
       order_reviews   99224        7
            products   32951        9
             sellers    3095        4
         geolocation 1000163        5
category_translation      71        2

Loaded 9 source tables.


### Observations from this execution

| Table | Rows | Columns |
|---|---:|---:|
| customers | 99,441 | 5 |
| orders | 99,441 | 8 |
| order_items | 112,650 | 7 |
| order_payments | 103,886 | 5 |
| order_reviews | 99,224 | 7 |
| products | 32,951 | 9 |
| sellers | 3,095 | 4 |
| geolocation | 1,000,163 | 5 |
| category_translation | 71 | 2 |

All nine source tables loaded successfully and contain rows. Their row units differ. For example, order items represent products within orders, and payments represent payment records. We will inspect keys and relationships before combining tables.

Missing-value handling, duplicate checks, timestamp parsing, and join validation will be documented in later sections.

**Next checkpoint** — Inspect sample rows, column types, missing values, identifiers, and duplicate keys.

**Suggested commit message** — `feat: start Olist notebook and load source tables`.


## 2 Inspect the source data

Inspect the loaded tables before deciding how to clean or combine them. This section examines sample rows, column types, missing values, repeated records, identifiers, and relationship coverage.

The checks create summaries from the source tables. We will use their results to justify cleaning and join rules in the next checkpoint.


### 2.1 Inspect sample rows

`.head(2)` shows the first two rows of each DataFrame. This helps identify what a row represents and how fields are stored. These examples illustrate structure. They are too small to describe a table's full distribution.

`max_colwidth=40` shortens long values only in the printed display, keeping the saved notebook readable.


In [4]:
for table_name, dataframe in tables.items():
    print(f"Table: {table_name}")
    print(dataframe.head(2).to_string(index=False, max_colwidth=40))
    print()


Table: customers
                     customer_id               customer_unique_id customer_zip_code_prefix         customer_city customer_state
06b8999e2fba1a1fbc88172c00ba8bc7 861eff4711a542e4b93843c6dd7febb0                    14409                franca             SP
18955e83d337fd6b2def6b18a428ac77 290c77bc529b7ac935b93aa66c333dc3                    09790 sao bernardo do campo             SP

Table: orders
                        order_id                      customer_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7 9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451 b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00      

### 2.2 Measure types and missing values

For every column, record its loaded dtype, number of missing values, percentage missing, and number of distinct non-missing values.

`.isna().sum()` counts missing values. Dividing by the row count and multiplying by 100 gives the missing percentage. `.nunique(dropna=True)` counts distinct observed values. This can help identify candidate identifiers, but uniqueness alone does not establish a column's business meaning.

Date fields may still be stored as strings. We will choose and explain their conversion after inspection.


In [5]:
profile_rows = []

for table_name, dataframe in tables.items():
    for column_name in dataframe.columns:
        column = dataframe[column_name]
        missing_count = int(column.isna().sum())
        profile_rows.append({
            "table": table_name,
            "column": column_name,
            "dtype": str(column.dtype),
            "missing_count": missing_count,
            "missing_percent": round(100 * missing_count / len(dataframe), 2),
            "unique_non_missing": int(column.nunique(dropna=True)),
        })

column_profile = pd.DataFrame(profile_rows)

print(column_profile.to_string(index=False))
print()
print("Columns with missing values:")
print(
    column_profile.loc[column_profile["missing_count"] > 0]
    .sort_values("missing_percent", ascending=False)
    .to_string(index=False)
)


               table                        column   dtype  missing_count  missing_percent  unique_non_missing
           customers                   customer_id     str              0             0.00               99441
           customers            customer_unique_id     str              0             0.00               96096
           customers      customer_zip_code_prefix  string              0             0.00               14994
           customers                 customer_city     str              0             0.00                4119
           customers                customer_state     str              0             0.00                  27
              orders                      order_id     str              0             0.00               99441
              orders                   customer_id     str              0             0.00               99441
              orders                  order_status     str              0             0.00                   8
 

### 2.3 Check repeated records and identifiers

`.duplicated().sum()` counts rows identical across every column after the first occurrence. `.duplicated(subset=key_columns).sum()` counts repeated key combinations after their first occurrence, even when other fields differ.

For order items, the pair `order_id` and `order_item_id` identifies a line item. For payments, inspect the pair `order_id` and `payment_sequential`. A shared order identifier can occur legitimately across several items or payments.

Inspect review identifiers as candidates that need investigation. A postal-code prefix can map to multiple geographic observations, and a stable customer identifier can appear on multiple order-level customer records. Repeated values in these fields need interpretation before choosing a deduplication rule.


In [6]:
duplicate_rows = []

for table_name, dataframe in tables.items():
    duplicate_rows.append({
        "table": table_name,
        "total_rows": len(dataframe),
        "exact_duplicate_rows": int(dataframe.duplicated().sum()),
    })

duplicate_summary = pd.DataFrame(duplicate_rows)

key_definitions = [
    ("customers", ["customer_id"], "expected unique"),
    ("customers", ["customer_unique_id"], "customer may recur"),
    ("orders", ["order_id"], "expected unique"),
    ("order_items", ["order_id", "order_item_id"], "expected unique"),
    ("order_payments", ["order_id", "payment_sequential"], "expected unique"),
    ("order_reviews", ["review_id"], "candidate to investigate"),
    ("order_reviews", ["order_id"], "order may have multiple reviews"),
    ("products", ["product_id"], "expected unique"),
    ("sellers", ["seller_id"], "expected unique"),
    ("geolocation", ["geolocation_zip_code_prefix"], "prefix may recur"),
    ("category_translation", ["product_category_name"], "expected unique"),
]

key_rows = []

for table_name, key_columns, interpretation in key_definitions:
    dataframe = tables[table_name]
    key_rows.append({
        "table": table_name,
        "key_columns": " + ".join(key_columns),
        "missing_key_rows": int(dataframe[key_columns].isna().any(axis=1).sum()),
        "duplicate_key_rows": int(dataframe.duplicated(subset=key_columns).sum()),
        "interpretation": interpretation,
    })

key_summary = pd.DataFrame(key_rows)

print("Exact duplicate rows:")
print(duplicate_summary.to_string(index=False))
print()
print("Identifier checks:")
print(key_summary.to_string(index=False))


Exact duplicate rows:
               table  total_rows  exact_duplicate_rows
           customers       99441                     0
              orders       99441                     0
         order_items      112650                     0
      order_payments      103886                     0
       order_reviews       99224                     0
            products       32951                     0
             sellers        3095                     0
         geolocation     1000163                261831
category_translation          71                     0

Identifier checks:
               table                   key_columns  missing_key_rows  duplicate_key_rows                  interpretation
           customers                   customer_id                 0                   0                 expected unique
           customers            customer_unique_id                 0                3345              customer may recur
              orders                      ord

### 2.4 Check relationship coverage before joining

For each relationship, `.isin()` checks whether a non-missing child identifier occurs in its parent table. We count missing child identifiers separately from observed identifiers with no matching parent.

Coverage checks measure whether matches exist. They do not establish safe join cardinality. Parent uniqueness and repeated records must also be considered before a merge.

Finally, count orders without matching item, payment, or review records. This shows where an inner join could remove orders. Record status and timing will need examination before deciding how to handle these cases.


In [7]:
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "category_translation", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]

relationship_rows = []

for child_table, child_column, parent_table, parent_column in relationships:
    child_values = tables[child_table][child_column]
    parent_values = tables[parent_table][parent_column].dropna()
    unmatched = child_values.notna() & ~child_values.isin(parent_values)
    relationship_rows.append({
        "child": f"{child_table}.{child_column}",
        "parent": f"{parent_table}.{parent_column}",
        "missing_child_values": int(child_values.isna().sum()),
        "unmatched_child_rows": int(unmatched.sum()),
    })

relationship_summary = pd.DataFrame(relationship_rows)

coverage_rows = []
order_ids = tables["orders"]["order_id"]

for related_table in ["order_items", "order_payments", "order_reviews"]:
    coverage_rows.append({
        "related_table": related_table,
        "orders_without_records": int(
            (~order_ids.isin(tables[related_table]["order_id"])).sum()
        ),
    })

order_coverage = pd.DataFrame(coverage_rows)

print("Relationship coverage:")
print(relationship_summary.to_string(index=False))
print()
print("Orders without related records:")
print(order_coverage.to_string(index=False))


Relationship coverage:
                             child                                     parent  missing_child_values  unmatched_child_rows
                orders.customer_id                      customers.customer_id                     0                     0
              order_items.order_id                            orders.order_id                     0                     0
            order_items.product_id                        products.product_id                     0                     0
             order_items.seller_id                          sellers.seller_id                     0                     0
           order_payments.order_id                            orders.order_id                     0                     0
            order_reviews.order_id                            orders.order_id                     0                     0
    products.product_category_name category_translation.product_category_name                   610                    13
c

### 2.5 Findings and implications for cleaning

| Observation | Result from the downloaded CSVs | Implication for the next section |
|---|---|---|
| Stable customer identity | 96,096 unique customer identifiers across 99,441 customer records | Use the stable identity to connect histories, while preserving valid order-level customer records |
| Exact duplicates | 261,831 additional identical geolocation rows; zero in the other eight tables | Investigate and document geolocation deduplication before preparing a postal-code lookup |
| Review identifiers | 814 repeated review-ID occurrences and 551 repeated order-ID occurrences after their first appearance | Inspect review relationships and dates before selecting an order-level aggregation or record-selection rule |
| Missing review text | 88.34 percent of titles and 58.70 percent of messages are missing | Preserve transaction records without text; any later NLP analysis needs its own eligible cohort |
| Missing product category | 610 products have no category | Decide how to represent unknown categories without losing otherwise useful product records |
| Untranslated categories | 13 products have a non-missing category absent from the translation table | Inspect those category names and define an explicit label fallback |
| Missing order timestamps | 160 approvals, 1,783 carrier handovers, and 2,965 customer deliveries | Examine order status before interpreting missing lifecycle events |
| Incomplete order coverage | 775 orders lack items, 1 lacks payments, and 768 lack reviews | Measure the effect of join and eligibility choices before filtering orders |
| Geographic coverage | 278 customer records and 7 seller records have postal prefixes absent from geolocation | Keep optional location enrichment from unintentionally removing otherwise usable records |

The duplicate-key counts describe additional rows after the first occurrence. They are not counts of distinct affected customers, orders, or reviews.

All identifiers marked as expected unique passed this inspection, including the composite item and payment keys. The checked order, customer, product, and seller references all matched their parent identifiers. Coverage of optional category translations and geographic lookups needs explicit handling.

We have created diagnostic summaries and retained the loaded source tables. The next checkpoint will inspect problematic records and establish cleaning rules with measured before-and-after effects.

**Suggested commit message** — `feat: inspect Olist schemas missing data and keys`.


## 3 Clean and prepare the source tables

### 3.1 Convert timestamps and audit order timing

Convert the source timestamp strings to pandas datetimes so that dates can be compared and durations calculated. First distinguish dates that were already missing from values that fail parsing.

Create `prepared_tables` as a working mapping. Copy each table being changed before converting its date columns, preserving the loaded source tables in `tables`. The current checkpoint changes only the timestamp representation in orders, order items, and reviews.

Missing lifecycle dates need interpretation using order status. A canceled or still-processing order may legitimately lack later events. Keep missing dates as `NaT`, pandas' missing-datetime value, and inspect chronological inconsistencies before deciding record eligibility.


In [8]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

prepared_tables = tables.copy()
conversion_rows = []

for table_name, columns in date_columns.items():
    prepared_tables[table_name] = tables[table_name].copy()
    for column_name in columns:
        original = tables[table_name][column_name]
        converted = pd.to_datetime(
            original,
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce",
        )
        newly_missing = original.notna() & converted.isna()
        conversion_rows.append({
            "table": table_name,
            "column": column_name,
            "missing_before": int(original.isna().sum()),
            "parse_failures": int(newly_missing.sum()),
            "missing_after": int(converted.isna().sum()),
            "new_dtype": str(converted.dtype),
        })
        prepared_tables[table_name][column_name] = converted

timestamp_conversion = pd.DataFrame(conversion_rows)
print(timestamp_conversion.to_string(index=False))


        table                        column  missing_before  parse_failures  missing_after      new_dtype
       orders      order_purchase_timestamp               0               0              0 datetime64[us]
       orders             order_approved_at             160               0            160 datetime64[us]
       orders  order_delivered_carrier_date            1783               0           1783 datetime64[us]
       orders order_delivered_customer_date            2965               0           2965 datetime64[us]
       orders order_estimated_delivery_date               0               0              0 datetime64[us]
  order_items           shipping_limit_date               0               0              0 datetime64[us]
order_reviews          review_creation_date               0               0              0 datetime64[us]
order_reviews       review_answer_timestamp               0               0              0 datetime64[us]


### 3.2 Examine missing dates by order status

Count missing date values separately for each recorded status. `.isna()` creates boolean columns and grouping then summing counts the missing values within each status.

These statuses describe the recorded dataset. In future predictive features, eventual status will require an availability check against the historical prediction date.


In [9]:
orders_prepared = prepared_tables["orders"]

lifecycle_missing = orders_prepared[date_columns["orders"]].isna()
lifecycle_missing.insert(0, "order_status", orders_prepared["order_status"])

missing_by_status = lifecycle_missing.groupby("order_status").sum()
missing_by_status.insert(
    0,
    "orders",
    orders_prepared.groupby("order_status").size(),
)

print(missing_by_status.to_string())
print()
print("Observed purchase timestamp coverage:")
print(orders_prepared["order_purchase_timestamp"].agg(["min", "max"]).to_string())


              orders  order_purchase_timestamp  order_approved_at  order_delivered_carrier_date  order_delivered_customer_date  order_estimated_delivery_date
order_status                                                                                                                                                 
approved           2                         0                  0                             2                              2                              0
canceled         625                         0                141                           550                            619                              0
created            5                         0                  5                             5                              5                              0
delivered      96478                         0                 14                             2                              8                              0
invoiced         314                         0      

### 3.3 Flag inconsistent event sequences

For each pair of actual events, compare timestamps only when both are present. Count cases where the later event is recorded before the earlier one.

Keep flags in a separate table keyed by order identifier. These checks identify records for investigation. The effect on each modeling task will be assessed before applying exclusions.


In [10]:
event_pairs = [
    ("approval_before_purchase", "order_purchase_timestamp", "order_approved_at"),
    ("carrier_before_approval", "order_approved_at", "order_delivered_carrier_date"),
    ("delivery_before_carrier", "order_delivered_carrier_date", "order_delivered_customer_date"),
    ("delivery_before_purchase", "order_purchase_timestamp", "order_delivered_customer_date"),
]

order_timing_flags = orders_prepared[["order_id"]].copy()
timing_rows = []

for flag_name, earlier_column, later_column in event_pairs:
    earlier = orders_prepared[earlier_column]
    later = orders_prepared[later_column]
    both_present = earlier.notna() & later.notna()
    inconsistent = both_present & (later < earlier)
    order_timing_flags[flag_name] = inconsistent
    timing_rows.append({
        "check": flag_name,
        "orders_with_both_events": int(both_present.sum()),
        "inconsistent_orders": int(inconsistent.sum()),
    })

timing_summary = pd.DataFrame(timing_rows)
flag_columns = [pair[0] for pair in event_pairs]
orders_with_timing_flags = int(order_timing_flags[flag_columns].any(axis=1).sum())

print(timing_summary.to_string(index=False))
print()
print(f"Distinct orders with at least one timing flag: {orders_with_timing_flags}")


                   check  orders_with_both_events  inconsistent_orders
approval_before_purchase                    99281                    0
 carrier_before_approval                    97644                 1359
 delivery_before_carrier                    96475                   23
delivery_before_purchase                    96476                    0

Distinct orders with at least one timing flag: 1382


### Timestamp checkpoint results

Converted eight date columns in three working tables. There were 0 non-missing source values that failed the specified timestamp format.

| check | orders_with_both_events | inconsistent_orders |
| --- | --- | --- |
| approval_before_purchase | 99281 | 0 |
| carrier_before_approval | 97644 | 1359 |
| delivery_before_carrier | 96475 | 23 |
| delivery_before_purchase | 96476 | 0 |

Across these checks, 1382 distinct orders have at least one timing flag. A record can trigger several checks, so the individual counts should not be added to obtain a distinct-order total.

Missing lifecycle dates remain visible in the status summary above. Keep the measured flags available for later eligibility and feature decisions. Counts, dates, and statuses should be interpreted together.

**Next checkpoint** — Investigate review multiplicity and create an explicit, documented review selection or aggregation policy.

**Suggested commit message** — `feat: parse Olist timestamps and audit order timing`.
